# Author data-quality rules on Spark with redibis

Discover quality rules on a **full partition**, check them on the **next** partition, curate, and merge them into the table's **data contract**. Then see what a month-later refresh would change.

Full tutorial: `docs/tutorials/QUALITY_AUTHORING.md`. Needs `pip install -e ".[ge,spark]"` and Java 17+.

On a real cluster, use your notebook's Spark session and replace `partition(...)` with something like `spark.table("sales.transactions").where("txn_date = '2026-09-24'")`.

In [ ]:
import logging
from pyspark.sql import SparkSession, functions as F
from redibis.quality.authoring import QualityAuthor, SNAPSHOT_RULES

logging.getLogger("great_expectations").setLevel(logging.ERROR)
spark = (SparkSession.builder.master("local[4]").appName("redibis-quality-authoring")
         .config("spark.ui.enabled", "false").getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
TABLE = "sales.transactions"

A synthetic daily partition of a transactions table (1M rows). Skip this on a real cluster.

In [ ]:
def partition(day, seed, rows=1_000_000, currencies=("EGP", "USD", "EUR")):
    pick = F.element_at(F.array(*[F.lit(c) for c in currencies]),
                        (F.col("id") % len(currencies) + 1).cast("int"))
    return (spark.range(rows)
            .withColumn("txn_id", F.format_string(f"TXN-{day.replace('-', '')}-%09d", F.col("id")))
            .withColumn("amount", (F.rand(seed) * 5000).cast("decimal(12,2)"))
            .withColumn("currency", pick)
            .withColumn("channel", F.when(F.col("id") % 10 == 0, None).otherwise(F.lit("app")))
            .withColumn("txn_date", F.lit(day).cast("date"))
            .drop("id"))

## 1. Discover rules on one full partition

Discovery and validation run inside Spark. `output_dir` is the same store the CLI uses with `--output-dir` (use `QualityAuthor.from_s3(...)` for a shared S3/MinIO store).

In [ ]:
qa = QualityAuthor(TABLE, output_dir="./reports")
draft = qa.author(partition("2026-09-24", seed=1))
print(draft.summary())

## 2. Is it true tomorrow?

`validate()` runs the rules on other data and writes nothing.

In [ ]:
next_day = partition("2026-09-25", seed=2, rows=950_000)
before = draft.validate(next_day)
print(f"as discovered: {before.rules_passed}/{before.rules_total}")

## 3. Curate

Drop the exact-snapshot statistics (mean, median, row count…), drop the per-day date column, widen the remaining numeric ranges by ±10 %.

In [ ]:
draft.drop(rule_types=SNAPSHOT_RULES)
draft.drop(columns=["txn_date"])
draft.relax(0.1)
after = draft.validate(next_day)
print(f"after curation: {after.status} {after.rules_passed}/{after.rules_total}")
print(draft.summary())

## 4. Save as a run, check the diff, merge into the contract

You can stop after `save()` and do the rest from the CLI: `redibis quality-run show / diff / edit / merge`.

In [ ]:
run = qa.save(draft, note="authored on 2026-09-24, checked on 2026-09-25")
print(qa.diff(run.run_id))
print("merged: v" + qa.merge(run.run_id).upsert.version_after)

## 5. A month later

A new currency appears. The diff shows it before anything changes in the contract.

In [ ]:
october = qa.author(partition("2026-10-24", seed=3, currencies=("EGP", "USD", "EUR", "SAR")), log=None)
october.drop(rule_types=SNAPSHOT_RULES)
october.drop(columns=["txn_date"])
october.relax(0.1)
refresh = qa.save(october, note="October refresh")
print(qa.diff(refresh.run_id))

## 6. Review and merge from the CLI

```bash
redibis quality-run show  sales.transactions --output-dir ./reports
redibis quality-run edit  sales.transactions --run <run_id> --drop-rule in_set --note "why" --output-dir ./reports
redibis quality-run merge sales.transactions --run <run_id> --output-dir ./reports
```

No shared store? `draft.to_yaml("rules.yaml")` here, then `redibis quality-run import sales.transactions rules.yaml` where the CLI runs.